In [2]:
import numpy as np
from pathlib import Path
import sys

for path in (Path.cwd(), Path.cwd() / "certificates" / "empirical_laws", Path.cwd().parent / "certificates" / "empirical_laws"):
    if (path / "notebook_setup.py").exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

import notebook_setup
from lyapunov import bisection, has_lyapunov
from utils import dask_parallel_map


EPSILONS = np.linspace(0.05, 0.95, 10)
L_VALUES = [1.0, 100.0, 1000.0]
KAPPA_VALUES = [2, 100, 1000]
N_WORKERS_RANGE = [2, 3, 4]

ETA_GRID_RESOLUTION = 10
GAMMA_GRID_RESOLUTION = 10
TEST_IMPROVEMENT = 1 - 1e-2
BISECTION_TOL = 1e-7
MOSEK_SOLVE_KWARGS = notebook_setup.MOSEK_STRICT_SOLVE_KWARGS
SDPA_SOLVE_KWARGS = notebook_setup.SDPA_HIGH_PRECISION_SOLVE_KWARGS
DASK_SCHEDULER = "processes"
DASK_NUM_WORKERS = 14
STOP_ON_FIRST_FAILURE = True


def gamma_star(eps, Ls, mus, n_workers):
    return (2 * n_workers / np.sum(Ls + mus)) * ((1 - np.sqrt(eps)) / (1 + np.sqrt(eps)))


def econtrol_rate(*, eta, gamma, lyap_kwargs, solver, solve_kwargs):
    rho, _, _ = bisection(
        0.0,
        1.0,
        BISECTION_TOL,
        has_lyapunov,
        eta=float(eta),
        gamma=float(gamma),
        solver=solver,
        solve_kwargs=solve_kwargs,
        **lyap_kwargs,
    )
    return rho



def eta_gamma_grid(*, rho_target, L_tuple, kappa_tuple, n_workers, scale_info):
    raw_Ls = np.array(L_tuple, dtype=float)
    raw_mus = raw_Ls / np.array(kappa_tuple, dtype=float)
    raw_eta_max = 2.0 * n_workers / np.sum(raw_Ls + raw_mus)
    eta_grid = np.linspace(0.0, raw_eta_max, ETA_GRID_RESOLUTION)

    # Under common scaling L_i, mu_i -> L_i/s, mu_i/s, eta is unchanged
    # while gamma is multiplied by s.
    gamma_max = raw_eta_max * scale_info["scale_down"]
    mu_bar = float(np.mean(scale_info["mus"]))
    L_bar = float(np.mean(scale_info["Ls"]))
    # Any candidate with rate <= rho_target must contract the exact-compression
    # x-subspace; this necessary band excludes degenerate values such as gamma=0.
    q = np.sqrt(np.clip(float(rho_target), 0.0, 1.0))
    gamma_lo = (1.0 - q) / mu_bar
    gamma_hi = min(gamma_max, (1.0 + q) / L_bar)
    if gamma_lo > gamma_hi:
        return eta_grid, np.array([])
    gamma_grid = np.linspace(gamma_lo, gamma_hi, GAMMA_GRID_RESOLUTION)
    return eta_grid, gamma_grid


def confirm_improvement(*, rho_imp, eta_candidate, gamma_candidate, lyap_kwargs):
    rho_candidate = econtrol_rate(
        eta=eta_candidate,
        gamma=gamma_candidate,
        lyap_kwargs=lyap_kwargs,
        solver="MOSEK",
        solve_kwargs=MOSEK_SOLVE_KWARGS,
    )
    if rho_candidate is None or rho_candidate > rho_imp:
        return None

    sdpa_info = notebook_setup.warning_aware_solve(
        has_lyapunov,
        rho_imp,
        eta=float(eta_candidate),
        gamma=float(gamma_candidate),
        solver="SDPA",
        solve_kwargs=SDPA_SOLVE_KWARGS,
        **lyap_kwargs,
    )
    if not sdpa_info["ok_clean"]:
        return None
    return rho_candidate


def check_normalized_config(*, L_tuple, kappa_tuple, eps, n_workers, scale_info):
    Ls = scale_info["Ls"]
    mus = scale_info["mus"]
    lyap_kwargs = {
        "delta": 1 - eps,
        "n_workers": n_workers,
        "mus": mus,
        "Ls": Ls,
        "method": "EControl",
        "use_simplified_lyapunov": False,
        "homogenous": False,
        "log_det_iterations": 0,
    }
    gamma_ref = gamma_star(eps, Ls, mus, n_workers)

    rho_star = econtrol_rate(
        eta=0.0,
        gamma=gamma_ref,
        lyap_kwargs=lyap_kwargs,
        solver="MOSEK",
        solve_kwargs=MOSEK_SOLVE_KWARGS,
    )
    if rho_star is None:
        return False, f"{notebook_setup.scale_label(scale_info)} reference rate not certified"

    rho_imp = rho_star * TEST_IMPROVEMENT
    eta_grid, gamma_grid = eta_gamma_grid(
        rho_target=rho_imp,
        L_tuple=L_tuple,
        kappa_tuple=kappa_tuple,
        n_workers=n_workers,
        scale_info=scale_info,
    )
    if gamma_grid.size == 0:
        return None, f"{notebook_setup.scale_label(scale_info)} rho_ref={rho_star:.10f}, empty necessary gamma band"


    for eta_candidate in eta_grid:
        for gamma_candidate in gamma_grid:
            ok_imp, _, _ = has_lyapunov(
                rho_imp,
                eta=float(eta_candidate),
                gamma=float(gamma_candidate),
                solver="MOSEK",
                solve_kwargs=MOSEK_SOLVE_KWARGS,
                **lyap_kwargs,
            )
            if not ok_imp:
                continue

            confirmed = confirm_improvement(
                rho_imp=rho_imp,
                eta_candidate=float(eta_candidate),
                gamma_candidate=float(gamma_candidate),
                lyap_kwargs=lyap_kwargs,
            )
            if confirmed is None:
                continue

            rho_candidate = confirmed
            return True, (
                f"{notebook_setup.scale_label(scale_info)} improved rate certified: "
                f"rho_ref={rho_star:.10f}, rho_imp={rho_imp:.10f}, "
                f"rho_candidate={rho_candidate:.10f}, eta={float(eta_candidate):.8g}, "
                f"gamma={float(gamma_candidate):.8g}"
            )

    return None, f"{notebook_setup.scale_label(scale_info)} rho_ref={rho_star:.10f}"


def verify_config(args):
    L_tuple, kappa_tuple, eps, n_workers = args
    scale_info = notebook_setup.scaled_problem_data_for_case(L_tuple, kappa_tuple)
    improved, message = check_normalized_config(
        L_tuple=L_tuple,
        kappa_tuple=kappa_tuple,
        eps=eps,
        n_workers=n_workers,
        scale_info=scale_info,
    )
    if improved:
        return f"FAIL: n={n_workers} L={L_tuple} kappa={kappa_tuple} Eps={eps:.6f} | {message}"
    if improved is False:
        return (
            f"FAIL: n={n_workers} L={L_tuple} kappa={kappa_tuple} Eps={eps:.6f} | "
            f"uncertified reference: {message}"
        )
    return None

def build_configs(n_workers):
    worker_configs = notebook_setup.worker_L_kappa_configs(
        L_VALUES,
        KAPPA_VALUES,
        n_workers,
        dedup_permutations=True,
    )
    return [
        (L_cfg, kappa_cfg, float(eps), n_workers)
        for (L_cfg, kappa_cfg) in worker_configs
        for eps in EPSILONS
    ]


def run_rigorous_check(n_workers=2):
    configs = build_configs(n_workers)
    print(f"--- Verification Suite: EControl Tuning, n={n_workers} ---")
    print(f"Checking {len(configs)} configurations with {DASK_NUM_WORKERS} workers...")
    print(f"Grid size per config: {ETA_GRID_RESOLUTION} x {GAMMA_GRID_RESOLUTION}")

    results = dask_parallel_map(
        verify_config,
        configs,
        scheduler=DASK_SCHEDULER,
        num_workers=DASK_NUM_WORKERS,
    )
    failures = [msg for msg in results if msg]
    if not failures:
        print("ALL CHECKS PASSED\n")
        return []

    for msg in failures:
        print(msg)
    print(f"FAILURES={len(failures)}\n")
    return failures


if __name__ == "__main__":
    total_failures = []
    for n_workers in N_WORKERS_RANGE:
        failures = run_rigorous_check(n_workers=n_workers)
        total_failures.extend(failures)
        if failures and STOP_ON_FIRST_FAILURE:
            break

    if total_failures:
        print(f"TOTAL_FAILURES={len(total_failures)}")
        raise SystemExit(1)
    print("ALL ECONTROL TUNING CHECKS PASSED")


--- Verification Suite: EControl Tuning, n=2 ---
Checking 450 configurations with 14 workers...
Grid size per config: 10 x 10


compute: 100%|██████████| 450/450 [01:38<00:00,  4.58it/s]


ALL CHECKS PASSED

--- Verification Suite: EControl Tuning, n=3 ---
Checking 1650 configurations with 14 workers...
Grid size per config: 10 x 10


compute: 100%|██████████| 1650/1650 [13:06<00:00,  2.10it/s]


ALL CHECKS PASSED

--- Verification Suite: EControl Tuning, n=4 ---
Checking 4950 configurations with 14 workers...
Grid size per config: 10 x 10


compute: 100%|██████████| 4950/4950 [1:19:23<00:00,  1.04it/s]


ALL CHECKS PASSED

ALL ECONTROL TUNING CHECKS PASSED
